In [27]:
using BenchmarkTools, Plots, Random, LaTeXStrings
using StaticArrays, LinearAlgebra, DelimitedFiles

include("methods.jl")
include("lsr.jl")
include("mgrit.jl")
include("lyapunov.jl")

# 3D systems plotting helper
Plots.plot(u::Vector{SVector{3, T}}; kwargs...) where T<:Real = Plots.plot(eachrow(svectors2mat(u))...; kwargs...)
Plots.plot!(u::Vector{SVector{3, T}}; kwargs...) where T<:Real = Plots.plot!(Plots.plot!(), eachrow(svectors2mat(u))...; kwargs...)

function savearray(array, fname)
    open(fname, "w") do f
        writedlm(fname, array)
    end
end


savearray (generic function with 1 method)

In [28]:
function conv_rate(residuals; len=5)
    r = residuals[1:findlast(isfinite, residuals)]
    # return exp((log(residuals[end]) - log(residuals[end-5]))/5)
    return prod(r[end-len+1:end] ./ r[end-len:end-1])^(1/len)
end

# functions to initialize trajectories
function FMG_cycle_init!(fine_grid::TimeGrid, relax, nl; relax_kwargs...)
    grids = construct_hierarchy(fine_grid, nl)

    skip_down_cycle!(grids)
    sequential_solve!(grids[end])
    up_cycle!(grids, relax; relax_kwargs...)

    return fine_grid
end

function gaussian_init!(fine_grid::TimeGrid, relax, nl; relax_kwargs...)
    nt = num_time_points(fine_grid.u)
    fine_grid.u .= 1e1randn(SVector{3}, nt) .+ fill(SVector(0.0, 0.0, ρ), nt)

    return fine_grid
end

function perturbed_init!(fine_grid::TimeGrid, noise_size)
    nt = num_time_points(fine_grid.u)
    sequential_solve!(fine_grid)
    fine_grid.u .+= noise_size .* randn(SVector{3}, nt)

    return fine_grid
end

# implement cycling strategy and return hierarchy and residual history
function cycling(cycle::Function, fine_grid::TimeGrid, nl, iters=20, convrate_len=5)
    grids = construct_hierarchy(fine_grid, nl)

    residuals = fill(NaN, iters+1)
    residuals[1] = spacetimenorm(rel_residual(grids[1]))

    # @show residuals[1]
    for k ∈ 2:iters+1
        isfinite(residuals[k-1]) || break
        if residuals[k-1] < 1e-15
            break
        end
        cycle(grids, residuals[k-1], k-1)
        residuals[k] = spacetimenorm(rel_residual(grids[1]))
        @show residuals[k]
    end
    @show residuals[findlast(isfinite, residuals)]
    @show conv_rate(residuals; len=convrate_len)

    return grids, residuals
end


cycling (generic function with 3 methods)

In [29]:
# asymptotic convergence study for V-cycling with different α values (no overrelaxation, no regridding)

relax_kwargs=(iters=2, overrelax=2, maxiters=1, damping=1.0, alg=:my_gauss_newton)

function lorenz_α_study(Nt, Tfλ, alphas; nl=7, m=2, cycle_iters=40, sys=lorenz, relax_kwargs...) 

    Random.seed!(12345)
    Tλ = log(10)/0.9
    Tf = Tfλ*Tλ
    h = Tf / (Nt-1)

    u0 = @SVector [0.1, 0.1, ρ]
    u0 += 1e-1randn(SVector{3})
    u = fill(u0, Nt+1) 

    fg = TimeGrid(deepcopy(u), sys, theta_method(0.5), h, m)
    perturbed_init!(fg, 1e-1)

    g_init = deepcopy(fg)
    u_init = copy(fg.u)

    @show spacetimenorm(rel_residual(fg))

    all_residuals = Vector{Float64}[]

    for α ∈ alphas
        fg = TimeGrid(deepcopy(u_init), sys, theta_method(0.5), h, m)

        grids, res = cycling(fg, nl, cycle_iters) do grids, rnorm, iter
            down_cycle!(grids, lsr!; relax_kwargs..., α=α)
            lsr!(grids[end]; relax_kwargs..., α=α)
            up_cycle!(grids, lsr!; relax_kwargs..., α=α)
            # F_up!(grids, lsr!, 1; relax_kwargs..., α=α)
        end
        push!(all_residuals, res)
    end

    return g_init, fg, reduce(hcat, all_residuals)

end

niters=20
alphas = [1e3, 1e1, 1e0]
savearray(alphas, "data/alpha_study/alphas.txt")

g_init, g_final, all_residuals_16 = lorenz_α_study(2^14, 16., alphas; cycle_iters=niters, relax_kwargs...)
savearray(all_residuals_16, "data/alpha_study/residuals_16.txt")

g_init, g_final, all_residuals_64 = lorenz_α_study(2^16, 64., alphas; cycle_iters=niters, relax_kwargs...)
savearray(all_residuals_64, "data/alpha_study/residuals_64.txt")

g_init, g_final, all_residuals_256 = lorenz_α_study(2^18, 256., alphas; cycle_iters=niters, relax_kwargs...)
savearray(all_residuals_256, "data/alpha_study/residuals_256.txt")

plot(0:niters, all_residuals_16; label=alphas' .|> α->L"\alpha = %$α", color=[1:8...]', yaxis=:log10)

spacetimenorm(rel_residual(fg)) = 0.06512461286783543
residuals[k] = 1.0018847601714172e-5
residuals[k] = 5.753223790626874e-6
residuals[k] = 5.689454947373869e-6
residuals[k] = 5.92069096840821e-6
residuals[k] = 6.149386236589697e-6
residuals[k] = 6.379351140003488e-6
residuals[k] = 6.610755945972088e-6
residuals[k] = 6.863182049464361e-6
residuals[k] = 7.297344440468527e-6
residuals[k] = 7.76318134374151e-6
residuals[k] = 8.263337665469608e-6
residuals[k] = 8.799795562482308e-6
residuals[k] = 9.374470669214265e-6
residuals[k] = 9.989230587550685e-6
residuals[k] = 1.0645974170099452e-5
residuals[k] = 1.134667847152947e-5
residuals[k] = 1.2093427809915542e-5
residuals[k] = 1.2888432976793996e-5
residuals[k] = 1.3734045079014717e-5
residuals[k] = 1.4632766526442786e-5
residuals[findlast(isfinite, residuals)] = 1.4632766526442786e-5
conv_rate(residuals; len = convrate_len) = 1.0656834150639558
residuals[k] = 8.6248897451385e-6
residuals[k] = 7.699022840671378e-6
residuals[k] = 5.59950096

In [30]:
# asymptotic convergence study for V-cycling with different number of levels (no overrelaxation, no regridding)
relax_kwargs=(iters=2, overrelax=2, maxiters=1, damping=1.0, alg=:my_gauss_newton, α=1e0)
function lorenz_nl_study(Nt, Tfλ, nls; m=2, cycle_iters=40, sys=lorenz, relax_kwargs...) 

    Random.seed!(1234)
    Tλ = log(10)/0.9
    Tf = Tfλ*Tλ
    h = Tf / (Nt-1)

    u0 = @SVector [0.1, 0.1, ρ]
    u0 += 1e-1randn(SVector{3})
    u = fill(u0, Nt+1) 

    fg = TimeGrid(deepcopy(u), sys, theta_method(0.5), h, m)
    # fg = TimeGrid(deepcopy(u), sys, euler, h, m)
    perturbed_init!(fg, 1e-1)

    g_init = deepcopy(fg)
    u_init = copy(fg.u)

    @show spacetimenorm(rel_residual(fg))

    all_residuals = Vector{Float64}[]

    for nl ∈ nls
        fg = TimeGrid(deepcopy(u_init), sys, theta_method(0.5), h, m)

        grids, res = cycling(fg, nl, cycle_iters) do grids, rnorm, iter
            down_cycle!(grids, lsr!; relax_kwargs...)
            lsr!(grids[end]; relax_kwargs...)
            up_cycle!(grids, lsr!; relax_kwargs...)
            # F_up!(grids, lsr!, 1; relax_kwargs...)
        end
        push!(all_residuals, res)
    end

    return g_init, fg, reduce(hcat, all_residuals)

end

niters = 20
maxnl = 7
g_init, g_final, all_residuals_16 = lorenz_nl_study(2^14, 16., 1:2:maxnl; cycle_iters=niters, relax_kwargs...)
savearray(all_residuals_16, "data/nl_study/residuals_16.txt")

g_init, g_final, all_residuals_64 = lorenz_nl_study(2^16, 64., 1:2:maxnl; cycle_iters=niters, relax_kwargs...)
savearray(all_residuals_64, "data/nl_study/residuals_64.txt")

g_init, g_final, all_residuals_256 = lorenz_nl_study(2^18, 256., 1:2:maxnl; cycle_iters=niters, relax_kwargs...)
savearray(all_residuals_256, "data/nl_study/residuals_256.txt")

# plot(all_residuals')
plot(0:niters, all_residuals_16; label=(1:2:maxnl)' .|> nl->L"N_{\ell} = %$nl", color=[1:8...]', yaxis=:log10)
# end

spacetimenorm(rel_residual(fg)) = 0.14753539243280023
residuals[k] = 0.015323330720271233
residuals[k] = 0.01155408942815028
residuals[k] = 0.008984300974678432
residuals[k] = 0.007107884929345016
residuals[k] = 0.005765357446368931
residuals[k] = 0.004789316063337689
residuals[k] = 0.004059782034647942
residuals[k] = 0.003497961917716939
residuals[k] = 0.0030532842453184234
residuals[k] = 0.0026931225247784625
residuals[k] = 0.002395984703906821
residuals[k] = 0.002147284888336475
residuals[k] = 0.0019367806884383386
residuals[k] = 0.0017570258080367566
residuals[k] = 0.0016024277314170322
residuals[k] = 0.0014686642472866176
residuals[k] = 0.0013523139293165499
residuals[k] = 0.001250615783162206
residuals[k] = 0.0011613082306449814
residuals[k] = 0.0010825178358794764
residuals[findlast(isfinite, residuals)] = 0.0010825178358794764
conv_rate(residuals; len = convrate_len) = 0.9245519559007974
residuals[k] = 0.0008862679761171833
residuals[k] = 0.0005281116803635574
residuals[k] = 0.

In [31]:
# FMG solve convergence study for different numbers of levels
relax_kwargs=(iters=2, overrelax=1, maxiters=1, α=1e-1, alg=:my_gauss_newton, initial_boundary=false)

function lorenz_study(Nt, Tfλ, nls; m=2, nl_fmg=7, cycle_iters=40, sys=lorenz, relax_kwargs...) 
    Tλ = log(10)/0.9
    Tf = Tfλ*Tλ
    h = Tf / (Nt-1)

    Random.seed!(12345)

    # u0 = @SVector [0.1, 0.1, ρ]
    u0 = @SVector [3., 5., 7.]
    # u0 = @SVector [1.0, 0.0, 1.0]
    # u0 += 1e1randn(SVector{3})
    # u0 += 1e-1randn(SVector{3})
    u = fill(u0, Nt+1) 

    fg = TimeGrid(deepcopy(u), sys, theta_method(0.5), h, m)
    FMG_cycle_init!(fg, f_relax!, nl_fmg)
    # gaussian_init!(fg, lsr!, nl_fmg)

    g_init = deepcopy(fg)
    u_init = copy(fg.u)

    @show spacetimenorm(rel_residual(fg))

    all_residuals = Vector{Float64}[]

    for nl_solve ∈ nls
        fg = TimeGrid(deepcopy(u_init), sys, theta_method(0.5), h, m)

        grids, res = cycling(fg, nl_solve, cycle_iters) do grids, rnorm, iter
            αk = max(relax_kwargs[:α], (1e3rnorm)^2)
            # αk = 10.
            # damping = 1.0 + max(0., 0.5tanh(-log(1e3rnorm)))
            damping = 1.0 + max(0., 0.5tanh(-(3 + log10(rnorm))))
            # damping = 1.0
            # init_boundary = rnorm > 1e-3
            # init_boundary = false
            init_boundary = iter == 1

            # damping = rnorm < 1e-4 ? 1.5 : 1.0
            # damping = 1.0
            if iter == 1 # solve coarse grid
                skip_down_cycle!(grids)
                # sequential_solve!(grids[end])
            else
                regrid!(grids[1])
                down_cycle!(grids, lsr!; relax_kwargs..., α=αk, damping=damping, initial_boundary=init_boundary)
                lsr!(grids[end]; relax_kwargs..., α=αk, damping=damping, initial_boundary=init_boundary)
            end
            # up_cycle!(grids, lsr!; relax_kwargs..., α=αk, damping=damping, init_boundary=init_boundary)
            F_up!(grids, lsr!, 1; relax_kwargs..., α=αk, damping=damping, initial_boundary=init_boundary)
        end
        push!(all_residuals, res)
    end

    return g_init, fg, reduce(hcat, all_residuals)

end

function do_trials(N, args...; kwargs...)
    residuals = Vector{Float64}[]
    for t ∈ 1:N
        g_init, g_final, t_residuals = lorenz_study(args...; kwargs...)
        push!(residuals, vec(t_residuals))
    end
    return g_init, g_final, sum(residuals)./N
end

nlmax = 7
niters = 20
nls = nlmax:nlmax
g_init, g_final, all_residuals_16 = lorenz_study(2^14, 16., nls; cycle_iters=niters, sys=lorenz, relax_kwargs...)
savearray(all_residuals_16, "data/fmg_study/residuals_16.txt")
savearray(g_init.u, "data/fmg_study/init_16.txt")
savearray(g_final.u, "data/fmg_study/final_16.txt")

g_init, g_final, all_residuals_64 = lorenz_study(2^16, 64., nls; cycle_iters=niters, sys=lorenz, relax_kwargs...)
savearray(all_residuals_64, "data/fmg_study/residuals_64.txt")
savearray(g_init.u, "data/fmg_study/init_64.txt")
savearray(g_final.u, "data/fmg_study/final_64.txt")

g_init, g_final, all_residuals_256 = lorenz_study(2^18, 256., nls; cycle_iters=niters, sys=lorenz, relax_kwargs...)
savearray(all_residuals_256, "data/fmg_study/residuals_256.txt")
savearray(g_init.u, "data/fmg_study/init_256.txt")
savearray(g_final.u, "data/fmg_study/final_256.txt")

plot(all_residuals_16; yaxis=:log10)
plot!(all_residuals_64)
plot!(all_residuals_256)
# end

spacetimenorm(rel_residual(fg)) = 0.19179181005417453
residuals[k] = 0.002481253624095809
residuals[k] = 0.0005084133932977887
residuals[k] = 0.00012967961293544885
residuals[k] = 8.572663262485079e-5
residuals[k] = 4.063312219928865e-6
residuals[k] = 7.427399793004662e-8
residuals[k] = 1.3655986423519504e-9
residuals[k] = 1.1791835587510748e-11
residuals[k] = 2.1959884242567027e-13
residuals[k] = 8.117163481360066e-15
residuals[k] = 4.4254337885346317e-16
residuals[findlast(isfinite, residuals)] = 4.4254337885346317e-16
conv_rate(residuals; len = convrate_len) = 0.022647684651254107
spacetimenorm(rel_residual(fg)) = 0.2212594248802575
residuals[k] = 0.0025869780670114886
residuals[k] = 0.0006046072118244184
residuals[k] = 0.0001520723279877443
residuals[k] = 0.00012269229597076145
residuals[k] = 2.7953394248344405e-5
residuals[k] = 1.1800378116410102e-6
residuals[k] = 1.8514621768238613e-8
residuals[k] = 1.8732856746867931e-10
residuals[k] = 2.275659593482213e-12
residuals[k] = 4.1602

In [32]:
# FMG solve convergence study for gaussian init
relax_kwargs=(iters=2, overrelax=2, maxiters=1, α=1e-1, alg=:my_gauss_newton)

function lorenz_study_rand(Nt, Tfλ, nls; m=2, nl_fmg=7, cycle_iters=40, sys=lorenz, relax_kwargs...) 

    Random.seed!(12345)
    Tλ = log(10)/0.9
    Tf = Tfλ*Tλ
    h = Tf / (Nt-1)

    # these don't really matter
    u0 = @SVector [0.1, 0.1, ρ]
    u0 += 1e-1randn(SVector{3})
    u = fill(u0, Nt+1)

    fg = TimeGrid(deepcopy(u), sys, theta_method(0.5), h, m)
    gaussian_init!(fg, lsr!, nl_fmg)

    g_init = deepcopy(fg)
    u_init = copy(fg.u)

    @show spacetimenorm(rel_residual(fg))

    all_residuals = Vector{Float64}[]

    for nl_solve ∈ nls
        fg = TimeGrid(deepcopy(u_init), sys, theta_method(0.5), h, m)

        grids, res = cycling(fg, nl_solve, cycle_iters) do grids, rnorm, iter
            αk = max(relax_kwargs[:α], (1e3rnorm)^2)
            damping = 1.0 + max(0., 0.5tanh(-(3 + log10(rnorm))))

            # skip the first down-cycle, since the initial guess is random anyway
            if iter == 1
                skip_down_cycle!(grids)
            else
                regrid!(grids[1])
                down_cycle!(grids, lsr!; relax_kwargs..., α=αk, damping=damping)
                # down_cycle!(grids, f_relax!)
            end

            lsr!(grids[end]; relax_kwargs..., α=αk, damping=damping)
            # up_cycle!(grids, lsr!; relax_kwargs..., α=αk, damping=damping)
            F_up!(grids, lsr!, 1; relax_kwargs..., α=αk, damping=damping)
        end
        push!(all_residuals, res)
    end

    return g_init, fg, reduce(hcat, all_residuals)

end

nlmax = 6
niters = 40
nls = nlmax:nlmax
g_init, g_final, all_residuals_16 = lorenz_study_rand(2^14, 16., nls; cycle_iters=niters, sys=lorenz, relax_kwargs...)
savearray(all_residuals_16, "data/random_study/residuals_16.txt")
savearray(g_init.u, "data/random_study/init_16.txt")
savearray(g_final.u, "data/random_study/final_16.txt")
g_init, g_final, all_residuals_64 = lorenz_study_rand(2^16, 64., nls; cycle_iters=niters, sys=lorenz, relax_kwargs...)
savearray(all_residuals_64, "data/random_study/residuals_64.txt")
savearray(g_init.u, "data/random_study/init_64.txt")
savearray(g_final.u, "data/random_study/final_64.txt")
g_init, g_final, all_residuals_256 = lorenz_study_rand(2^18, 256., nls; cycle_iters=niters, sys=lorenz, relax_kwargs...)
savearray(all_residuals_256, "data/random_study/residuals_256.txt")
savearray(g_init.u, "data/random_study/init_256.txt")
savearray(g_final.u, "data/random_study/final_256.txt")

plot(all_residuals_16; yaxis=:log10)
plot!(all_residuals_64; yaxis=:log10)
plot!(all_residuals_256; yaxis=:log10)
# end

spacetimenorm(rel_residual(fg)) = 19.11646429410351
residuals[k] = 0.01057451673998406
residuals[k] = 0.008485267094382156
residuals[k] = 0.006999184128473971
residuals[k] = 0.006944798290786522
residuals[k] = 0.006689947604396506
residuals[k] = 0.005973160116444599
residuals[k] = 0.004790236823735015
residuals[k] = 0.0031866161003927195
residuals[k] = 0.002151833837622187
residuals[k] = 0.0013564665006215615
residuals[k] = 0.0008999928515556224
residuals[k] = 0.0005706822654857842
residuals[k] = 0.0002913705206241234
residuals[k] = 6.404270179987733e-5
residuals[k] = 7.920766979110855e-7
residuals[k] = 2.0873150957686927e-9
residuals[k] = 5.832063462152583e-11
residuals[k] = 2.622156795237411e-13
residuals[k] = 9.735801005835539e-16
residuals[findlast(isfinite, residuals)] = 9.735801005835539e-16
conv_rate(residuals; len = convrate_len) = 0.0068608892456964434
spacetimenorm(rel_residual(fg)) = 19.116482616937134
residuals[k] = 0.014314327537121339
residuals[k] = 0.012429671036270715
r

In [33]:
# Now do an lsr convergence plot
Random.seed!(1010)

function res_component(r, Γ, j)
    r_Γ = project_onbasis.(r, Γ, false)
    return [γ[j] for γ ∈ r_Γ]
end

fine_args = (α=1e0, maxiters=1, overrelax=1, alg=:my_gauss_newton, damping=1.00, initial_boundary=true)

m = 2
g0 = TimeGrid(copy(g_init.u[1:2^14+1]), g_init.sys, g_init.disc, g_init.Δt, g_init.m)
sequential_solve!(g0)
u0 = copy(g0.u)
Ψmn, Ψpl, Γ, Θ = lyap_vectors(u0, g0.Δt, g0.disc, g0.sys)

skip = m
noise = randn(g0.Nt + 1)
g0.u .+= randn(SVector{3}, g0.Nt+1)
u_perturbed = copy(g0.u)

# plot portion of trajectory
plot_portion = 2^13:m:2^13+2^9
u0_portion = u0[plot_portion]
g0_portion = u_perturbed[plot_portion]

f_relax!(g0)
e0 = (norm.(g0.u .- u0))[skip:skip:end]
r0 = norm.(rel_residual(g0))[skip:skip:end]
ru0 = res_component(residual(g0)[skip:skip:end], Ψpl[2:end][skip:skip:end], 1)
ru0 .*= maximum(r0) ./ maximum(ru0)

t0 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
lsr!(g0; fine_args...)

e1 = (norm.(g0.u .- u0))[skip:skip:end]
r1 = norm.(rel_residual(g0))[skip:skip:end]

t1 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
lsr!(g0; iters=9, fine_args...)

g10_portion = g0.u[plot_portion]

e10 = (norm.(g0.u .- u0))[skip:skip:end]
r10 = norm.(rel_residual(g0))[skip:skip:end]
t10 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
lsr!(g0; iters=90, fine_args...)

e100 = (norm.(g0.u .- u0))[skip:skip:end]
r100 = norm.(rel_residual(g0))[skip:skip:end]
t100 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
lsr!(g0; iters=900, fine_args...)

g1000_portion = g0.u[plot_portion]

e1000 = (norm.(g0.u .- u0))[skip:skip:end]
r1000 = norm.(rel_residual(g0))[skip:skip:end]
t1000 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]

residuals_lsr = [norm(r0, Inf), norm(r1, Inf), norm(r10, Inf), norm(r100, Inf), norm(r1000, Inf)]

Lyapunov exponents λ = [0.8412553584521699, -0.020101382390504872, -14.542704945259795]


5-element Vector{Float64}:
 0.5775981773835963
 0.17128363014544454
 0.03691002833772917
 0.007265188885452916
 0.0013478184982226406

In [34]:
# pgfplotsx()
plot(u0_portion; line=(0.5, :dash, :black), alpha=0.75, label="shadow")
plot!(g0_portion; color=1, linewidth=0.75, linealpha=1.0, label="init. guess")
plot!(g10_portion; color=2, linewidth=0.75, label="10 iter. LSR")
# plot!(g1000_portion; color=3, linewidth=0.75, label="1000 iter. LSR")
plot!(; axis=:none, ticks=:none, legend=:bottomright, size=(300,300))

# savefig("../figures/1lvl/smoothing_zoom_in.pdf")
# gr()

In [35]:
p = plot(; ylabel=L"\|r(t)\|_{2}", xlabel=L"T_{\lambda}", legend=:bottomright, ylim=(1e-5, 1e0),
           xticks=[0, 4, 8, 12, 16])
lw = 1.0
plot!(p, t0, r0; yaxis=:log10, label="init.", linewidth=lw)
plot!(p, t1, r1; yaxis=:log10, label="iter. 1", linewidth=lw)
plot!(p, t10, r10; yaxis=:log10, label="iter. 10", linewidth=lw)
plot!(p, t100, r100; yaxis=:log10, label="iter. 100", linewidth=lw)
plot!(p, t1000, r1000; yaxis=:log10, label="iter. 1000", linewidth=lw)
# savefig(p, "data/lsr_converge_1lvl.pdf")

# plot!(p, t0, e0; yaxis=:log10, label="init.", linewidth=lw)
# plot!(p, t1, e1; yaxis=:log10, label="iter. 1", linewidth=lw)
# plot!(p, t10, e10; yaxis=:log10, label="iter. 10", linewidth=lw)
# plot!(p, t100, e100; yaxis=:log10, label="iter. 100", linewidth=lw)
# plot!(p, t1000, e1000; yaxis=:log10, label="iter. 1000", linewidth=lw)

┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid neg

In [36]:
nt = length(r0)
sig = exp(g0.Δt * 0.9)
nmodes = nt
modes = zeros(nt, nmodes)
rates = zeros(nmodes)
for k = 1:nmodes
    ω = ((1:nt) .- 1/2).*(k*π)./nt
    modes[:, k] .= sin.(ω) |> v->v./norm(v)
    rates[k] = (2 + 2cos(k*π/nt)) * sig^2 / (1 + sig^2)^2
end

modes
plot(modes'*ru0)
plot!(rates.^1 .* modes'*ru0)
plot!(rates.^10 .* modes'*ru0)

In [37]:
r1_pred = abs.(modes * (rates .* modes'*ru0))
r10_pred = abs.(modes * (rates.^10 .* modes'*ru0))
r100_pred = abs.(modes * (rates.^100 .* modes'*ru0))
r1000_pred = abs.(modes * (rates.^1000 .* modes'*ru0))

residuals_pred = norm.([r1_pred, r10_pred, r100_pred, r1000_pred], Inf)

p = plot(; ylabel=L"\|r(t)\|_{2}", xlabel=L"T_{\lambda}", ylim=(1e-5, 1e0),
           xticks=[0, 4, 8, 12, 16])
lw = 1.0
plot!(p, t0, r0; yaxis=:log10, label="init.", linewidth=lw)
# plot!(p, t1, r1; yaxis=:log10, label="iter. 1", linewidth=lw)
plot!(p, t10, r10; yaxis=:log10, label="iter. 10", linewidth=lw)
# plot!(p, t100, r100; yaxis=:log10, label="iter. 100", linewidth=lw)
plot!(p, t1000, r1000; yaxis=:log10, label="iter. 1000", linewidth=lw)
# plot!(p, t1, r1_pred; yaxis=:log10, label="iter. 1 (predicted)", linewidth=lw)
# hline!(p, [residuals_pred[1]]; color=2, linestyle=:dash, label=:none)
hline!(p, [residuals_pred[2]]; color=2, linestyle=:dash, label=:none)
# hline!(p, [residuals_pred[3]]; color=4, linestyle=:dash, label=:none)
hline!(p, [residuals_pred[4]]; color=3, linestyle=:dash, label=:none)
# savefig(p, "data/lsr_converge_1lvl_bounds.pdf")
# plot!(p, t10, r10_pred; yaxis=:log10, label="iter. 10 (predicted)", linewidth=lw)
# plot!(p, t100, r100_pred; yaxis=:log10, label="iter. 100 (predicted)", linewidth=lw)
# plot!(p, t1000, r1000_pred; yaxis=:log10, label="iter. 1000 (predicted)", linewidth=lw)

┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105


In [38]:
# Now do a fcf convergence plot
Random.seed!(1010)

skip = m
g0.u .= u_perturbed
f_relax!(g0)
# plot(g0.u)
r0 = norm.(rel_residual(g0))[skip:skip:end]
t0 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
fcf_relax!(g0)
r1 = norm.(rel_residual(g0))[skip:skip:end]
t1 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
fcf_relax!(g0; iters=9)
r10 = norm.(rel_residual(g0))[skip:skip:end]
t10 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
fcf_relax!(g0; iters=90)
r100 = norm.(rel_residual(g0))[skip:skip:end]
t100 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
fcf_relax!(g0; iters=900)
r1000 = norm.(rel_residual(g0))[skip:skip:end]
t1000 = cumsum(dilate.(g0.η, g0.Δt) ./ log(10) .* 0.9)[skip:skip:end]
residuals_fcf = norm.([r0, r1, r10, r100, r1000], Inf)

p = plot(; ylabel=L"\|r(t)\|_{2}", xlabel=L"T_{\lambda}", ylim=(1e-3, 1e1),
           xticks=[0, 4, 8, 12, 16])
lw = 1.0
plot!(p, t0, r0; yaxis=:log10, label="init.", linewidth=lw)
plot!(p, t1, r1; yaxis=:log10, label="iter. 1", linewidth=lw)
plot!(p, t10, r10; yaxis=:log10, label="iter. 10", linewidth=lw)
plot!(p, t100, r100; yaxis=:log10, label="iter. 100", linewidth=lw)
plot!(p, t1000, r1000; yaxis=:log10, label="iter. 1000", linewidth=lw)
# savefig(p, "data/fcf_converge_1lvl.pdf")

┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid negative or zero value 0.0 found at series index 1 for log10 based yscale
└ @ Plots /Users/davarga/.julia/packages/Plots/GIume/src/utils.jl:105
┌ Warning: Invalid neg

In [39]:
# plot (2,∞)-norm of residuals
pgfplotsx()
plot(; axis=:log10, xticks=([1, 2, 10, 100, 1000], ["0", "1", "10", "100", "1000"]),
       xlabel="iteration", ylabel=L"\|r\|_{(2, \infty)}", size=(300,300), legend=:bottomleft)
plot!([1, 2, 11, 101, 1001], residuals_fcf; marker=:circle, label="FCF")
plot!([1, 2, 11, 101, 1001], residuals_lsr; marker=:circle, label="LSR")
plot!([2, 11, 101, 1001], residuals_pred; marker=:circle, label="LSR (theory)")
savefig("data/compare_converge_1lvl.pdf")
# gr()

"/Users/davarga/Projects/LSR_Reproducibility/data/compare_converge_1lvl.pdf"